In [1]:
#Packages to Import

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag

#Fun Progress Bar
from tqdm.notebook import tqdm

#Misc System (plotting etc)
import sys
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

import pickle
import warnings
warnings.filterwarnings('ignore')


#MCMC Sampliers and Related Utilities
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
%run MCMC_Sampliers.ipynb



#Plotting Libraries
import matplotlib.pyplot as plt
from numpy.linalg import norm



plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"]})



In [2]:
#Numerical Set up for AD Toy Model
#AD Toy Model

def MkAD_A_Mat(ModDim, curApar):
    """
    Generates an antisymmetric matric with the given Model Parameters
    """
    A = np.zeros([ModDim,ModDim])
    A[np.triu_indices(ModDim, k=1)] = curApar 
    #triu_indices returns the indices of all the above diagonal indicies
    return A - A.T

def getThA(ModDim, Apar, g, kappa):
    """
    Solves (A + kI) th = g  for theta given the specified model parameters determining A
    """
    A_p_kI = MkAD_A_Mat(ModDim, Apar)+ kappa*np.identity(ModDim)
    return np.linalg.solve(A_p_kI,g)

def mkDiagCov(vrs):
    return np.diag(vrs)


#Code to generate a random orthogonal matrix (uniform from O(n)) using the QR factorization of a Guassian matrix.
def rndm_orth_matrix(n):

    #Generate a random n x n matrix with i.i.d. normal entries
    A = np.random.randn(n, n)
    
    #Perform the QR factorization
    Q, R = np.linalg.qr(A)    
    
    return Q

In [3]:
#Example 1: From the Parallel MCMC paper

#Specifying Problem Parameters


#Model Dimension and Parameter Size

ModDm = 4
NumParms = int(ModDm*(ModDm -1)/2)


#The Forward Model entails solving for th(A) for any antisymmetric A where
# (A + kap I) th = g 
# so that th(A) = th_{k,g}

# g = (g0,g1, g2, g3)^T

g0 = .1
g1 = 0
g2 = 5
g3 = 2
g = np.transpose(np.array([g0,g1,g2,g3]))

# Coefficent of the `regularization/diffusion term'
kap = .05

#Specification `observed data' y0, y1
y0 = 4.601
y1 = 18.021

y = [y0,y1]

yData = np.array([y0,y1,0,0])

# `observation noise coefficent'

sig = 2

# Covariance of the 'prior' C = cov0[1^{-gam}, 2^[-gam],..., N^{-gam}]
# where N is the number of parameters in the model NumParms = 6 
#for number of enetries that need to be specified in A

cov0 = 5
gam = 1.5

# Specify Potential and Prior Covariance
# The Posterior is of the form
# mu(dA) = Z^{-1} \exp( -1/(2 sig^2) ( (y0 - th(A)(0))^2 +(y1 - th(A)(0))^2 ) mu_0(dA)
# where
# mu_0(dA) = Z^{-1}_0 \exp( - 1/2<C^{-1}A, A>)


Poty = lambda a : (2*sig**2)**(-1)*(norm(y - getThA(ModDm, a, g, kap)[0:2]))**2
CovDiag = [cov0* (j**(-gam)) for j in list(range(1,NumParms+1))]
Cov = mkDiagCov(CovDiag)

In [4]:
q00 = np.zeros(NumParms)
NumSamps0 = 1000
rho = .2
pSmp = 100
warmRun = MpCN(q00,NumParms,Cov,rho,Poty,pSmp,NumSamps0 +1)

In [7]:
#Generate Data Using MpCN


NumSamps = 500000 #samples per run

q0 = warmRun[NumSamps0]


rholoc = .8
rho = .1
pSmp = 100


mpCNMTMlocsampTr = locMpCNMTM(q0,NumParms,Cov,rholoc,Poty,pSmp,NumSamps +1,True)
mpCNMTMsampTr = MpCNBBMTM(q0,NumParms,Cov,rho,Poty,pSmp,NumSamps +1,True)

#Make Histogram


#Dimensions For Histogram Plot
R = 5
dr = .1

makeHistGrid(R, dr, mpCNMTMsampTr, NumParms,"MTM_N_" + str(NumSamps) + ".png", True)
makeHistGrid(R, dr, mpCNMTMlocsampTr, NumParms,"MTM_Loc_N_" + str(NumSamps) + ".png", True)

        

Rejection rate:
0.6775946448107104
Rejection rate:
0.5248149503700993
